# 3D U-Net Training — BraTS Brain Tumor Segmentation

Trains the custom 3D U-Net used in the paper on BraTS 2020 data.

**Architecture:** Encoder-decoder with GroupNorm, skip connections, trilinear upsampling.  
**Loss:** BCE + Dice  
**Input:** 4-channel MRI (FLAIR, T1, T1CE, T2), cropped to 128×128×128  
**Output:** 3-class binary masks (WT, TC, ET)  

Outputs written to `config.model_path`:
- `best_model.pth` — checkpoint with best validation Dice
- `last_epoch_model.pth` — checkpoint after final epoch
- `train_log.csv` — per-epoch train/val Dice and IoU

In [ ]:
from tqdm import tqdm
import os
import time
from random import randint

import gc
import numpy as np
import pandas as pd

import nibabel as nib
import matplotlib.pyplot as plt
from skimage.transform import resize

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

import albumentations as A
from albumentations import Compose

import warnings
warnings.simplefilter('ignore')

## Configuration

Update `root_dir` to your local data root and `model_path` for checkpoint output.

In [ ]:
class GlobalConfig:
    root_dir             = '/proj/arise/arise/suvodeep/'           # ← update to your data root
    model_path           = 'Models/Unet-3D_New/'
    train_root_dir       = root_dir + 'Data/BraTs2020/BraTS2020_TrainingData/MICCAI_BraTS2020_TrainingData'
    path_to_csv          = root_dir + 'Data/Processed_data/train_data.csv'
    pretrained_model_path = root_dir + model_path + 'last_epoch_model.pth'
    best_model_path      = root_dir + model_path + 'best_model.pth'
    train_logs_path      = root_dir + model_path + 'train_log.csv'
    seed                 = 55
    num_epochs           = 70
    batch_size           = 1
    num_workers          = 4
    lr                   = 1e-3
    num_folds            = 6       # k-fold cross-validation; fold 0 used as validation
    n_channels           = 24      # base channel width
    in_channels          = 4       # FLAIR, T1, T1CE, T2
    n_classes            = 3       # WT, TC, ET

    os.makedirs(root_dir + model_path, exist_ok=True)


def seed_everything(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


config = GlobalConfig()
seed_everything(config.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Data Preparation

Build `train_data.csv` with case paths and fold assignments (stratified k-fold).

In [ ]:
from sklearn.model_selection import StratifiedKFold

def build_train_csv(train_root_dir: str, n_folds: int = 6, seed: int = 55) -> pd.DataFrame:
    """Scan training directory, assign stratified k-fold labels, write CSV."""
    survival_info_df = pd.read_csv(os.path.join(train_root_dir, 'survival_info.csv'))
    name_mapping_df  = pd.read_csv(os.path.join(train_root_dir, 'name_mapping.csv'))
    name_mapping_df  = name_mapping_df.rename({'BraTS_2020_subject_ID': 'Brats20ID'}, axis=1)

    df = survival_info_df.merge(name_mapping_df, on='Brats20ID', how='left')
    df['path'] = df['Brats20ID'].apply(
        lambda x: os.path.join(train_root_dir, x)
    )

    # Stratify by age group as proxy label (common BraTS practice)
    df['Age_bin'] = pd.cut(df['Age'].fillna(df['Age'].median()), bins=4, labels=False)

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    df['fold'] = -1
    for fold, (_, val_idx) in enumerate(skf.split(df, df['Age_bin'])):
        df.loc[val_idx, 'fold'] = fold

    os.makedirs(os.path.dirname(config.path_to_csv), exist_ok=True)
    df.to_csv(config.path_to_csv, index=False)
    print(f'Saved {len(df)} cases to {config.path_to_csv}')
    print(df['fold'].value_counts().sort_index())
    return df


if not os.path.exists(config.path_to_csv):
    df = build_train_csv(config.train_root_dir, config.num_folds, config.seed)
else:
    df = pd.read_csv(config.path_to_csv)
    print(f'Loaded existing CSV: {len(df)} cases')

## Dataset and DataLoader

In [ ]:
class BratsDataset(Dataset):
    def __init__(self, df: pd.DataFrame, phase: str = 'train', is_resize: bool = False):
        self.df           = df
        self.phase        = phase
        self.augmentations = get_augmentations(phase)
        self.data_types   = ['_flair.nii', '_t1.nii', '_t1ce.nii', '_t2.nii']
        self.is_resize    = is_resize

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        id_       = self.df.loc[idx, 'Brats20ID']
        root_path = self.df.loc[self.df['Brats20ID'] == id_, 'path'].values[0]

        images = []
        for data_type in self.data_types:
            img_path = os.path.join(root_path, id_ + data_type)
            img = self.load_img(img_path)
            if self.is_resize:
                img = self.resize(img)
            img = self.normalize(img)
            images.append(img)

        img = np.stack(images)
        img = np.moveaxis(img, (0, 1, 2, 3), (0, 3, 2, 1))

        mask_path = os.path.join(root_path, id_ + '_seg.nii')
        mask = self.load_img(mask_path)
        if self.is_resize:
            mask = self.resize(mask)
            mask = np.clip(mask.astype(np.uint8), 0, 1).astype(np.float32)
        mask = self.preprocess_mask_labels(mask)

        augmented = self.augmentations(
            image=img.astype(np.float32),
            mask=mask.astype(np.float32)
        )
        img  = augmented['image']
        mask = augmented['mask']

        return {'Id': id_, 'image': img, 'mask': mask}

    def load_img(self, file_path):
        data = nib.load(file_path)
        return np.asarray(data.dataobj)

    def normalize(self, data: np.ndarray):
        data_min = np.min(data)
        return (data - data_min) / (np.max(data) - data_min + 1e-9)

    def resize(self, data: np.ndarray):
        return resize(data, (78, 120, 120), preserve_range=True)

    def preprocess_mask_labels(self, mask: np.ndarray):
        mask_WT = mask.copy()
        mask_WT[mask_WT == 1] = 1
        mask_WT[mask_WT == 2] = 1
        mask_WT[mask_WT == 4] = 1

        mask_TC = mask.copy()
        mask_TC[mask_TC == 1] = 1
        mask_TC[mask_TC == 2] = 0
        mask_TC[mask_TC == 4] = 1

        mask_ET = mask.copy()
        mask_ET[mask_ET == 1] = 0
        mask_ET[mask_ET == 2] = 0
        mask_ET[mask_ET == 4] = 1

        mask = np.stack([mask_WT, mask_TC, mask_ET])
        mask = np.moveaxis(mask, (0, 1, 2, 3), (0, 3, 2, 1))
        return mask


def get_augmentations(phase: str):
    """Horizontal flip for training; identity for validation."""
    transforms = []
    if phase == 'train':
        transforms.append(A.HorizontalFlip(p=0.5))
    return Compose(transforms, is_check_shapes=False)


def get_dataloader(
    dataset_cls,
    path_to_csv: str,
    phase: str,
    fold: int = 0,
    batch_size: int = 1,
    num_workers: int = 4,
) -> DataLoader:
    df = pd.read_csv(path_to_csv)
    if phase == 'train':
        split_df = df.loc[df['fold'] != fold].reset_index(drop=True)
        shuffle  = True
    else:
        split_df = df.loc[df['fold'] == fold].reset_index(drop=True)
        shuffle  = False
    print(f'{phase}: {len(split_df)} cases')
    ds = dataset_cls(split_df, phase)
    return DataLoader(
        ds,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
        shuffle=shuffle,
    )

## Loss Functions and Metrics

In [ ]:
def dice_coef_metric(probs: torch.Tensor, truth: torch.Tensor,
                     threshold: float = 0.5, eps: float = 1e-9) -> float:
    preds = (probs >= threshold).float()
    scores = []
    for i in range(preds.shape[0]):
        inter = 2.0 * (truth[i] * preds[i]).sum()
        union = truth[i].sum() + preds[i].sum()
        if truth[i].sum() == 0 and preds[i].sum() == 0:
            scores.append(1.0)
        else:
            scores.append(float((inter + eps) / union))
    return float(np.mean(scores))


def jaccard_coef_metric(probs: torch.Tensor, truth: torch.Tensor,
                        threshold: float = 0.5, eps: float = 1e-9) -> float:
    preds = (probs >= threshold).float()
    scores = []
    for i in range(preds.shape[0]):
        inter = (preds[i] * truth[i]).sum()
        union = preds[i].sum() + truth[i].sum() - inter + eps
        if truth[i].sum() == 0 and preds[i].sum() == 0:
            scores.append(1.0)
        else:
            scores.append(float((inter + eps) / union))
    return float(np.mean(scores))


class Meter:
    def __init__(self, threshold: float = 0.5):
        self.threshold  = threshold
        self.dice_scores: list = []
        self.iou_scores:  list = []

    def update(self, logits: torch.Tensor, targets: torch.Tensor):
        probs = torch.sigmoid(logits)
        self.dice_scores.append(dice_coef_metric(probs, targets, self.threshold))
        self.iou_scores.append(jaccard_coef_metric(probs, targets, self.threshold))

    def get_metrics(self):
        return float(np.mean(self.dice_scores)), float(np.mean(self.iou_scores))


class DiceLoss(nn.Module):
    def __init__(self, eps: float = 1e-9):
        super().__init__()
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        n = targets.size(0)
        prob  = torch.sigmoid(logits).view(n, -1)
        tgt   = targets.view(n, -1)
        inter = 2.0 * (prob * tgt).sum()
        union = prob.sum() + tgt.sum()
        return 1.0 - (inter + self.eps) / union


class BCEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce  = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        return self.bce(logits, targets) + self.dice(logits, targets)

## Model Architecture — 3D U-Net

In [ ]:
class DoubleConv(nn.Module):
    """(Conv3D → GroupNorm → ReLU) × 2"""
    def __init__(self, in_channels, out_channels, num_groups=8):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.GroupNorm(num_groups=num_groups, num_channels=out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.MaxPool3d(2, 2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        return self.encoder(x)


class Up(nn.Module):
    def __init__(self, in_channels, out_channels, trilinear=True):
        super().__init__()
        if trilinear:
            self.up = nn.Upsample(scale_factor=2, mode='trilinear', align_corners=True)
        else:
            self.up = nn.ConvTranspose3d(in_channels // 2, in_channels // 2,
                                         kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        dZ = x2.size(2) - x1.size(2)
        dY = x2.size(3) - x1.size(3)
        dX = x2.size(4) - x1.size(4)
        x1 = F.pad(x1, [dX // 2, dX - dX // 2,
                         dY // 2, dY - dY // 2,
                         dZ // 2, dZ - dZ // 2])
        return self.conv(torch.cat([x2, x1], dim=1))


class Out(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv3d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet3d(nn.Module):
    def __init__(self, in_channels, n_classes, n_channels):
        super().__init__()
        C = n_channels
        self.conv = DoubleConv(in_channels, C)
        self.enc1 = Down(C,     2 * C)
        self.enc2 = Down(2 * C, 4 * C)
        self.enc3 = Down(4 * C, 8 * C)
        self.enc4 = Down(8 * C, 8 * C)

        self.dec1 = Up(16 * C, 4 * C)
        self.dec2 = Up(8  * C, 2 * C)
        self.dec3 = Up(4  * C, C)
        self.dec4 = Up(2  * C, C)
        self.out  = Out(C, n_classes)

    def forward(self, x):
        x1 = self.conv(x)
        x2 = self.enc1(x1)
        x3 = self.enc2(x2)
        x4 = self.enc3(x3)
        x5 = self.enc4(x4)

        mask = self.dec1(x5, x4)
        mask = self.dec2(mask, x3)
        mask = self.dec3(mask, x2)
        mask = self.dec4(mask, x1)
        return self.out(mask)


model = UNet3d(
    in_channels=config.in_channels,
    n_classes=config.n_classes,
    n_channels=config.n_channels,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

## Trainer

In [ ]:
class Trainer:
    """
    Training loop for the 3D U-Net.

    Saves:
      - best_model.pth       when validation Dice improves
      - last_epoch_model.pth at the end of every epoch
      - train_log.csv        per-epoch metrics
    """

    def __init__(
        self,
        net: nn.Module,
        criterion: nn.Module,
        optimizer,
        scheduler,
        device: torch.device,
        num_epochs: int,
        best_model_path: str,
        last_model_path: str,
        train_logs_path: str,
    ):
        self.net             = net
        self.criterion       = criterion
        self.optimizer       = optimizer
        self.scheduler       = scheduler
        self.device          = device
        self.num_epochs      = num_epochs
        self.best_model_path = best_model_path
        self.last_model_path = last_model_path
        self.train_logs_path = train_logs_path
        self.best_dice       = 0.0
        self.log_rows        = []

    def run_epoch(self, loader: DataLoader, phase: str):
        is_train = phase == 'train'
        self.net.train(is_train)
        meter    = Meter()
        total_loss = 0.0

        with torch.set_grad_enabled(is_train):
            for batch in tqdm(loader, desc=phase, leave=False):
                images = batch['image'].permute(0, 4, 1, 2, 3).float().to(self.device)
                masks  = batch['mask'].permute(0, 4, 1, 2, 3).float().to(self.device)

                logits = self.net(images)
                loss   = self.criterion(logits, masks)

                if is_train:
                    self.optimizer.zero_grad()
                    loss.backward()
                    self.optimizer.step()

                total_loss += loss.item()
                meter.update(logits.detach().cpu(), masks.detach().cpu())

        avg_loss = total_loss / len(loader)
        dice, iou = meter.get_metrics()
        return avg_loss, dice, iou

    def run(self, train_loader: DataLoader, val_loader: DataLoader):
        for epoch in range(1, self.num_epochs + 1):
            t_start = time.time()

            train_loss, train_dice, train_iou = self.run_epoch(train_loader, 'train')
            val_loss,   val_dice,   val_iou   = self.run_epoch(val_loader,   'val')

            self.scheduler.step(val_dice)

            elapsed = time.time() - t_start
            lr_now  = self.optimizer.param_groups[0]['lr']

            print(
                f'Epoch {epoch:3d}/{self.num_epochs} | '
                f'Train loss {train_loss:.4f}  dice {train_dice:.4f}  iou {train_iou:.4f} | '
                f'Val   loss {val_loss:.4f}  dice {val_dice:.4f}  iou {val_iou:.4f} | '
                f'lr {lr_now:.6f} | {elapsed:.1f}s'
            )

            self.log_rows.append({
                'epoch':      epoch,
                'train_loss': train_loss, 'train_dice': train_dice, 'train_iou': train_iou,
                'val_loss':   val_loss,   'val_dice':   val_dice,   'val_iou':   val_iou,
                'lr':         lr_now,
            })
            pd.DataFrame(self.log_rows).to_csv(self.train_logs_path, index=False)

            torch.save(self.net, self.last_model_path)

            if val_dice > self.best_dice:
                self.best_dice = val_dice
                torch.save(self.net, self.best_model_path)
                print(f'  ✓ New best val Dice = {val_dice:.4f}  →  saved best_model.pth')

        print(f'\nTraining complete. Best val Dice = {self.best_dice:.4f}')

## Build DataLoaders, Optimizer, Scheduler and Train

In [ ]:
train_loader = get_dataloader(
    BratsDataset,
    config.path_to_csv,
    phase='train',
    fold=0,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
)
val_loader = get_dataloader(
    BratsDataset,
    config.path_to_csv,
    phase='val',
    fold=0,
    batch_size=config.batch_size,
    num_workers=config.num_workers,
)

criterion = BCEDiceLoss()
optimizer = Adam(model.parameters(), lr=config.lr)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='max',          # maximise val Dice
    factor=0.5,
    patience=7,
    min_lr=1e-6,
    verbose=True,
)

trainer = Trainer(
    net             = model,
    criterion       = criterion,
    optimizer       = optimizer,
    scheduler       = scheduler,
    device          = device,
    num_epochs      = config.num_epochs,
    best_model_path = config.best_model_path,
    last_model_path = config.pretrained_model_path,
    train_logs_path = config.train_logs_path,
)

trainer.run(train_loader, val_loader)

## Training Curves

In [ ]:
log_df = pd.read_csv(config.train_logs_path)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(log_df['epoch'], log_df['train_loss'], label='Train loss')
axes[0].plot(log_df['epoch'], log_df['val_loss'],   label='Val loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('BCE + Dice Loss'); axes[0].legend()

axes[1].plot(log_df['epoch'], log_df['train_dice'], label='Train Dice')
axes[1].plot(log_df['epoch'], log_df['val_dice'],   label='Val Dice')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice')
axes[1].set_title('Dice Score'); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(config.root_dir + config.model_path, 'training_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print(f'Best val Dice: {log_df["val_dice"].max():.4f}  (epoch {log_df["val_dice"].idxmax() + 1})')